# Federated Learning for Network Intrusion Detection (UNSW-NB15)
**Reproducible experiments for:** *AI-Based Network Intrusion Detection Using Federated Learning and Deep Learning*, Muhammad Awais Raza.

This notebook rebuilds the full study from scratch so that every number in the paper comes from one consistent, traceable run:

1. **Settings**: all experiment parameters in one place
2. **Data**: downloads UNSW-NB15 (official train/test split) and preprocesses it
3. **Non-IID partitioning**: Dirichlet split (alpha = 0.5) across 5 and 9 clients
4. **Models**: CNN, LSTM and hybrid CNN-LSTM
5. **Federated training**: FedAvg, written out explicitly
6. **Experiments**: model comparison (5 and 9 clients) and the hyperparameter study (H1, H2, H3)
7. **Charts and tables**: every figure the paper needs, plus a summary of the numbers to report

**How to run**
1. *Runtime > Change runtime type > T4 GPU*, then *Save*.
2. Choose *Runtime > Run all*. The full experiments take roughly 1–3 hours on a T4 GPU.
3. (Optional) To check the setup first, set `QUICK_TEST = True` for a 5-minute test run.

Results are saved to Google Drive after every experiment. If Colab disconnects, *Run all* again: finished experiments are loaded from Drive instead of being retrained.

## 1. Settings

In [ ]:
# ============================================================
# 1. SETTINGS  (the only cell you normally need to edit)
# ============================================================

QUICK_TEST = False       # False = full experiments. (True = 5-minute test with 10% data, 2 rounds)
SAVE_TO_DRIVE = True     # Save results to Google Drive so a disconnect doesn't lose them.
RESULTS_FOLDER = "FL_NIDS_results"   # Folder name inside "My Drive"

SEED = 42

# ---- Federated setup ----
CLIENT_COUNTS = [5, 9]   # Number of clients for the model comparison
DIRICHLET_ALPHA = 0.5    # Non-IID label skew (smaller = more skewed)
MIN_SAMPLES_PER_CLIENT = 200

# ---- Hyperparameter configurations (from the paper's Table / Fig. 23) ----
CONFIGS = {
    "H1": {"batch_size": 500, "local_epochs": 1, "rounds": 15},   # baseline
    "H2": {"batch_size": 128, "local_epochs": 5, "rounds": 14},   # small batch, multiple local epochs
    "H3": {"batch_size": 256, "local_epochs": 1, "rounds": 20},   # medium batch
}
MAIN_CONFIG = "H1"          # Configuration used for the CNN vs LSTM vs Hybrid comparison
HYPERPARAM_CLIENTS = 5      # Clients used for the H1/H2/H3 study on the Hybrid model

MODELS = ["CNN", "LSTM", "Hybrid"]
LEARNING_RATE = 1e-3
THRESHOLD = 0.5             # Decision threshold for attack vs normal

# ---- Quick-test overrides (applied automatically) ----
if QUICK_TEST:
    TRAIN_FRACTION = 0.10
    for c in CONFIGS.values():
        c["rounds"] = 2
        c["local_epochs"] = min(c["local_epochs"], 2)
    RESULTS_FOLDER = RESULTS_FOLDER + "_quicktest"
else:
    TRAIN_FRACTION = 1.0

print("QUICK_TEST:", QUICK_TEST)
for k, v in CONFIGS.items():
    print(k, v)

In [ ]:
# ---- Environment check, imports and reproducibility ----
import os, gc, time, json, random, glob, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, log_loss, confusion_matrix)
warnings.filterwarnings("ignore")

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

gpus = tf.config.list_physical_devices("GPU")
print("TensorFlow:", tf.__version__)
print("GPU:", gpus if gpus else "NONE  -> go to Runtime > Change runtime type > T4 GPU")

# ---- Where to save results ----
RESULTS_DIR = os.path.join("/content", RESULTS_FOLDER)
if SAVE_TO_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        RESULTS_DIR = os.path.join("/content/drive/MyDrive", RESULTS_FOLDER)
    except Exception as e:
        print("Drive not mounted, saving locally instead:", e)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.join(RESULTS_DIR, "figures"), exist_ok=True)
print("Results will be saved in:", RESULTS_DIR)

with open(os.path.join(RESULTS_DIR, "settings.json"), "w") as f:
    json.dump({"QUICK_TEST": QUICK_TEST, "SEED": SEED, "CLIENT_COUNTS": CLIENT_COUNTS,
               "DIRICHLET_ALPHA": DIRICHLET_ALPHA, "CONFIGS": CONFIGS, "MAIN_CONFIG": MAIN_CONFIG,
               "LEARNING_RATE": LEARNING_RATE, "TRAIN_FRACTION": TRAIN_FRACTION}, f, indent=2)

## 2. Data: UNSW-NB15
Uses the official training partition (175,341 records) and testing partition (82,332 records). Many copies of the dataset have the two file names swapped, so the notebook always trains on the larger file. The notebook tries a Kaggle download first. If that fails, it asks you to upload the two CSV files: `UNSW_NB15_training-set.csv` and `UNSW_NB15_testing-set.csv`.

In [ ]:
def find_csvs(root):
    files = glob.glob(os.path.join(root, "**", "*.csv"), recursive=True)
    tr = [f for f in files if "training-set" in os.path.basename(f).lower()]
    te = [f for f in files if "testing-set" in os.path.basename(f).lower()]
    return (tr[0], te[0]) if tr and te else (None, None)

train_csv, test_csv = find_csvs("/content")

if train_csv is None:
    try:
        import kagglehub
        path = kagglehub.dataset_download("mrwellsdavid/unsw-nb15")
        train_csv, test_csv = find_csvs(path)
        print("Downloaded from Kaggle:", path)
    except Exception as e:
        print("Kaggle download failed:", e)

if train_csv is None and SAVE_TO_DRIVE:
    train_csv, test_csv = find_csvs("/content/drive/MyDrive")

if train_csv is None:
    print("Please upload UNSW_NB15_training-set.csv and UNSW_NB15_testing-set.csv")
    from google.colab import files
    files.upload()
    train_csv, test_csv = find_csvs("/content")

assert train_csv and test_csv, "Dataset files not found."
print("Train file:", train_csv)
print("Test file :", test_csv)

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)
# Known issue: in many copies of UNSW-NB15 the training/testing file names are swapped.
# The official training partition has 175,341 records and the testing partition 82,332.
if len(train_df) < len(test_df):
    train_df, test_df = test_df, train_df
    print("NOTE: file names were swapped; using the larger file (175,341 rows) for training.")
print("Train shape:", train_df.shape, "| Test shape:", test_df.shape)
print("\nAttack categories (train):")
print(train_df["attack_cat"].value_counts())

### Preprocessing
* Drop `id`; keep `attack_cat` only for partitioning (the task is binary: `label` 0 = normal, 1 = attack).
* Label-encode the categorical features `proto`, `service` and `state`.
* Scale with MinMaxScaler fitted on the training set only (no test leakage).
* Reshape each record to `(n_features, 1)` so the same input feeds the Conv1D and LSTM layers.

In [ ]:
CAT_COLS = ["proto", "service", "state"]
DROP_COLS = ["id", "attack_cat", "label"]

full = pd.concat([train_df, test_df], axis=0)
for c in CAT_COLS:
    cats = {v: i for i, v in enumerate(sorted(full[c].astype(str).unique()))}
    train_df[c] = train_df[c].astype(str).map(cats)
    test_df[c] = test_df[c].astype(str).map(cats)

train_df["attack_cat"] = train_df["attack_cat"].fillna("Normal").astype(str).str.strip()
test_df["attack_cat"] = test_df["attack_cat"].fillna("Normal").astype(str).str.strip()

if TRAIN_FRACTION < 1.0:
    train_df = train_df.sample(frac=TRAIN_FRACTION, random_state=SEED).reset_index(drop=True)

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]
scaler = MinMaxScaler()
X_train = scaler.fit_transform(train_df[feature_cols].values.astype("float32"))
X_test = np.clip(scaler.transform(test_df[feature_cols].values.astype("float32")), 0, 1)

X_train = X_train.reshape(-1, len(feature_cols), 1).astype("float32")
X_test = X_test.reshape(-1, len(feature_cols), 1).astype("float32")
y_train = train_df["label"].values.astype("float32")
y_test = test_df["label"].values.astype("float32")
cat_train = train_df["attack_cat"].values

N_FEATURES = len(feature_cols)
del full  # free memory
print("Features:", N_FEATURES)
print("X_train:", X_train.shape, "| X_test:", X_test.shape)
print("Attack share  train: %.3f  test: %.3f" % (y_train.mean(), y_test.mean()))

## 3. Non-IID partitioning (Dirichlet, alpha = 0.5)
Each attack category (and *Normal*) is spread across clients using proportions drawn from a Dirichlet distribution. This gives every client a different mix of traffic, and rare categories such as *Worms* end up at only a few clients, as in real networks.

In [ ]:
def dirichlet_partition(categories, n_clients, alpha, seed, min_size):
    rng = np.random.default_rng(seed)
    classes = np.unique(categories)
    for attempt in range(100):
        parts = [[] for _ in range(n_clients)]
        for c in classes:
            idx = np.where(categories == c)[0]
            rng.shuffle(idx)
            props = rng.dirichlet(alpha * np.ones(n_clients))
            cuts = (np.cumsum(props) * len(idx)).astype(int)[:-1]
            for k, chunk in enumerate(np.split(idx, cuts)):
                parts[k].extend(chunk.tolist())
        if min(len(p) for p in parts) >= min_size:
            break
    return [np.array(sorted(p)) for p in parts]

PARTITIONS = {n: dirichlet_partition(cat_train, n, DIRICHLET_ALPHA, SEED, MIN_SAMPLES_PER_CLIENT)
              for n in sorted(set(CLIENT_COUNTS + [HYPERPARAM_CLIENTS]))}

def partition_stats(parts, name):
    rows = []
    for k, idx in enumerate(parts):
        vc = pd.Series(cat_train[idx]).value_counts()
        row = {"client": k + 1, "samples": len(idx), "attack_share": round(float(y_train[idx].mean()), 3)}
        row.update(vc.to_dict())
        rows.append(row)
    df = pd.DataFrame(rows).fillna(0)
    df.to_csv(os.path.join(RESULTS_DIR, f"partition_{name}.csv"), index=False)
    return df

for n, parts in PARTITIONS.items():
    df = partition_stats(parts, f"{n}clients")
    print(f"\n=== {n} clients (Dirichlet alpha = {DIRICHLET_ALPHA}) ===")
    display(df)

# Heatmap of category share per client (5 clients)
n_show = CLIENT_COUNTS[0]
df_show = partition_stats(PARTITIONS[n_show], f"{n_show}clients")
cat_cols = [c for c in df_show.columns if c not in ["client", "samples", "attack_share"]]
share = df_show[cat_cols].div(df_show[cat_cols].sum(axis=0), axis=1)
plt.figure(figsize=(10, 3.5))
plt.imshow(share.values, aspect="auto", cmap="Blues")
plt.colorbar(label="Share of category held by client")
plt.yticks(range(n_show), [f"Client {i+1}" for i in range(n_show)])
plt.xticks(range(len(cat_cols)), cat_cols, rotation=45, ha="right")
plt.title(f"Non-IID split across {n_show} clients (Dirichlet alpha = {DIRICHLET_ALPHA})")
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "figures", "partition_heatmap.png"), dpi=200)
plt.show()

## 4. Models
Architectures follow the paper: a 1D-CNN (filters 32/64/128 with down-sampling), a 3-layer LSTM with dropout, and a hybrid CNN-LSTM (Conv1D feature extraction followed by LSTM temporal modelling). All end in a sigmoid for binary classification.

In [ ]:
from tensorflow.keras import layers, models

def build_model(name, n_features=None):
    n_features = n_features or N_FEATURES
    inp = layers.Input(shape=(n_features, 1))
    if name == "CNN":
        x = layers.Conv1D(32, 3, padding="same", activation="relu")(inp)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(64, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Flatten()(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
    elif name == "LSTM":
        x = layers.LSTM(64, return_sequences=True)(inp)
        x = layers.Dropout(0.2)(x)
        x = layers.LSTM(64, return_sequences=True)(x)
        x = layers.Dropout(0.2)(x)
        x = layers.LSTM(32)(x)
        x = layers.Dropout(0.2)(x)
        x = layers.Dense(32, activation="relu")(x)
    elif name == "Hybrid":
        x = layers.Conv1D(64, 3, padding="same", activation="relu")(inp)
        x = layers.MaxPooling1D(2)(x)
        x = layers.Conv1D(128, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling1D(2)(x)
        x = layers.LSTM(64, return_sequences=True)(x)
        x = layers.LSTM(32)(x)
        x = layers.Dense(64, activation="relu")(x)
        x = layers.Dropout(0.3)(x)
    else:
        raise ValueError(name)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out, name=name)

for m in MODELS:
    mdl = build_model(m)
    print(f"{m}: {mdl.count_params():,} parameters ({mdl.count_params() * 4 / 1e6:.2f} MB as float32)")

## 5. Federated training (FedAvg)
Each round: the server sends the global weights to every client; each client trains locally for `local_epochs`; the server averages the returned weights in proportion to each client's number of samples. After every round the global model is evaluated on the held-out test set.

The client model is compiled once per experiment and its optimizer state is reset for each client. This is equivalent to a fresh local optimizer each round, and it avoids a TensorFlow memory leak that occurs when a model is re-compiled thousands of times.

In [ ]:
def evaluate(model):
    prob = model.predict(X_test, batch_size=2048, verbose=0).ravel()
    pred = (prob >= THRESHOLD).astype(int)
    return {
        "val_loss": float(log_loss(y_test, np.clip(prob, 1e-7, 1 - 1e-7))),
        "val_acc": float(accuracy_score(y_test, pred)),
        "precision": float(precision_score(y_test, pred, zero_division=0)),
        "recall": float(recall_score(y_test, pred, zero_division=0)),
        "f1": float(f1_score(y_test, pred, zero_division=0)),
        "roc_auc": float(roc_auc_score(y_test, prob)),
    }, prob

def reset_optimizer(model):
    opt = model.optimizer
    variables = opt.variables() if callable(opt.variables) else opt.variables
    for v in variables:
        name = (getattr(v, "path", None) or getattr(v, "name", "")).lower()
        if "learning_rate" in name:
            continue  # never reset the learning rate
        v.assign(tf.zeros_like(v))
    try:  # safety check: make sure the learning rate is still correct
        if abs(float(opt.learning_rate) - LEARNING_RATE) > 1e-12:
            opt.learning_rate = LEARNING_RATE
    except Exception:
        pass

def ram_gb():
    try:
        import psutil
        return psutil.Process(os.getpid()).memory_info().rss / 1e9
    except Exception:
        return float("nan")

def run_fedavg(model_name, n_clients, cfg_name):
    cfg = CONFIGS[cfg_name]
    tag = f"{model_name}_{n_clients}clients_{cfg_name}"
    csv_path = os.path.join(RESULTS_DIR, f"history_{tag}.csv")
    if os.path.exists(csv_path):
        print(f"[skip] {tag} already done, loading saved results.")
        return pd.read_csv(csv_path)

    tf.keras.backend.clear_session(); gc.collect()
    tf.random.set_seed(SEED)
    parts = PARTITIONS[n_clients]
    sizes = np.array([len(p) for p in parts], dtype="float64")
    client_data = [(X_train[idx], y_train[idx]) for idx in parts]   # built once per experiment
    global_model = build_model(model_name)
    worker = build_model(model_name)
    # Compile ONCE: re-compiling every round leaks memory in TensorFlow. Reset the optimizer per client instead.
    worker.compile(optimizer=tf.keras.optimizers.Adam(LEARNING_RATE),
                   loss="binary_crossentropy", metrics=["accuracy"])
    global_w = global_model.get_weights()
    rows = []
    print(f"\n=== {tag}: batch={cfg['batch_size']}, local_epochs={cfg['local_epochs']}, rounds={cfg['rounds']} ===")

    for r in range(1, cfg["rounds"] + 1):
        t0 = time.time()
        client_w, tr_loss, tr_acc = [], [], []
        for Xc, yc in client_data:
            worker.set_weights(global_w)
            reset_optimizer(worker)
            h = worker.fit(Xc, yc, epochs=cfg["local_epochs"],
                           batch_size=cfg["batch_size"], shuffle=True, verbose=0)
            client_w.append(worker.get_weights())
            tr_loss.append(h.history["loss"][-1]); tr_acc.append(h.history["accuracy"][-1])
        # FedAvg: weighted average by client sample count
        global_w = [np.sum([w[i] * s for w, s in zip(client_w, sizes)], axis=0) / sizes.sum()
                    for i in range(len(global_w))]
        del client_w
        round_time = time.time() - t0
        global_model.set_weights(global_w)
        metrics, _ = evaluate(global_model)
        row = {"round": r, "train_loss": float(np.average(tr_loss, weights=sizes)),
               "train_acc": float(np.average(tr_acc, weights=sizes)), "round_time_s": round_time, **metrics}
        rows.append(row)
        gc.collect()
        print(f"round {r:2d} | train_acc {row['train_acc']:.4f} | val_acc {row['val_acc']:.4f} | "
              f"P {row['precision']:.4f} R {row['recall']:.4f} F1 {row['f1']:.4f} | {round_time:.1f}s | RAM {ram_gb():.1f} GB")

    df = pd.DataFrame(rows)
    df.insert(0, "model", model_name); df.insert(1, "clients", n_clients); df.insert(2, "config", cfg_name)
    df.to_csv(csv_path, index=False)
    global_model.save(os.path.join(RESULTS_DIR, f"model_{tag}.keras"))
    return df

## 6. Experiments
**6a. Model comparison**: CNN vs LSTM vs Hybrid with the main configuration, for 5 and 9 clients.
**6b. Hyperparameter study**: the Hybrid model with H1, H2 and H3.
**6c. Fair comparison**: CNN and LSTM also trained with H2.

In [ ]:
RESULTS = {}
start = time.time()

# 6a. Model comparison
for n in CLIENT_COUNTS:
    for m in MODELS:
        RESULTS[(m, n, MAIN_CONFIG)] = run_fedavg(m, n, MAIN_CONFIG)

# 6b. Hyperparameter study on the Hybrid model (H1, H2, H3)
for cfg_name in CONFIGS:
    key = ("Hybrid", HYPERPARAM_CLIENTS, cfg_name)
    if key not in RESULTS:
        RESULTS[key] = run_fedavg("Hybrid", HYPERPARAM_CLIENTS, cfg_name)

# 6c. Fair comparison: CNN and LSTM with configuration H2 as well
for m in ["CNN", "LSTM"]:
    key = (m, HYPERPARAM_CLIENTS, "H2")
    if key not in RESULTS:
        RESULTS[key] = run_fedavg(m, HYPERPARAM_CLIENTS, "H2")

print(f"\nAll experiments finished in {(time.time() - start) / 60:.1f} minutes.")
all_hist = pd.concat(RESULTS.values(), ignore_index=True)
all_hist.to_csv(os.path.join(RESULTS_DIR, "all_histories.csv"), index=False)

## 7. Charts and tables

In [ ]:
FIG = os.path.join(RESULTS_DIR, "figures")
COLORS = {"CNN": "#1f4e79", "LSTM": "#c55a11", "Hybrid": "#70ad47"}

def save(name):
    plt.tight_layout(); plt.savefig(os.path.join(FIG, name), dpi=200); plt.show()

def per_round(metric, n, ylabel, fname):
    plt.figure(figsize=(7, 4))
    for m in MODELS:
        df = RESULTS[(m, n, MAIN_CONFIG)]
        plt.plot(df["round"], df[metric], marker="o", label=m, color=COLORS[m])
    plt.xlabel("Communication round"); plt.ylabel(ylabel)
    plt.title(f"{ylabel} per round ({n} clients, {MAIN_CONFIG})"); plt.grid(alpha=0.3); plt.legend()
    save(fname)

n0 = CLIENT_COUNTS[0]
per_round("val_acc", n0, "Validation accuracy", "val_accuracy.png")
per_round("val_loss", n0, "Validation loss", "val_loss.png")
per_round("train_acc", n0, "Training accuracy", "train_accuracy.png")
per_round("train_loss", n0, "Training loss", "train_loss.png")
per_round("precision", n0, "Precision", "precision_rounds.png")
per_round("recall", n0, "Recall", "recall_rounds.png")
per_round("f1", n0, "F1-score", "f1_rounds.png")

# Final-round precision / recall / F1 bars
final = pd.DataFrame([RESULTS[(m, n0, MAIN_CONFIG)].iloc[-1] for m in MODELS])
x = np.arange(len(MODELS)); w = 0.25
plt.figure(figsize=(7, 4))
for i, met in enumerate(["precision", "recall", "f1"]):
    plt.bar(x + (i - 1) * w, final[met], w, label=met.capitalize())
plt.xticks(x, MODELS); plt.ylim(0, 1.05); plt.ylabel("Score")
plt.title(f"Final-round detection metrics ({n0} clients)"); plt.legend(); plt.grid(axis="y", alpha=0.3)
save("final_metrics_bar.png")

# Training time per round
times = [RESULTS[(m, n0, MAIN_CONFIG)]["round_time_s"].mean() for m in MODELS]
plt.figure(figsize=(6, 4))
plt.bar(MODELS, times, color=[COLORS[m] for m in MODELS])
for i, t in enumerate(times): plt.text(i, t, f"{t:.1f}s", ha="center", va="bottom")
plt.ylabel("Mean time per round (s)"); plt.title(f"Training time per round ({n0} clients)")
save("training_time.png")

# 5 vs 9 clients
if len(CLIENT_COUNTS) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    for ax, met, lab in [(axes[0], "val_acc", "Validation accuracy"), (axes[1], "val_loss", "Validation loss")]:
        for m in MODELS:
            for n, ls in zip(CLIENT_COUNTS, ["-", "--"]):
                df = RESULTS[(m, n, MAIN_CONFIG)]
                ax.plot(df["round"], df[met], ls, color=COLORS[m], label=f"{m}, {n} clients")
        ax.set_xlabel("Communication round"); ax.set_ylabel(lab); ax.grid(alpha=0.3)
    axes[0].legend(fontsize=8)
    fig.suptitle("Effect of the number of clients")
    save("clients_comparison.png")

# Hyperparameter study: F1 vs time
hp = pd.DataFrame([{"config": c, **RESULTS[("Hybrid", HYPERPARAM_CLIENTS, c)].iloc[-1].to_dict(),
                    "mean_time": RESULTS[("Hybrid", HYPERPARAM_CLIENTS, c)]["round_time_s"].mean()}
                   for c in CONFIGS])
fig, ax1 = plt.subplots(figsize=(6.5, 4))
ax1.bar(hp["config"], hp["f1"] * 100, color=["#4472c4", "#70ad47", "#ed7d31"])
for i, v in enumerate(hp["f1"] * 100): ax1.text(i, v, f"{v:.1f}%", ha="center", va="bottom")
ax1.set_ylabel("F1-score (%)"); ax1.set_ylim(max(0, hp["f1"].min() * 100 - 5), 100)
ax2 = ax1.twinx(); ax2.plot(hp["config"], hp["mean_time"], "o--", color="gray"); ax2.set_ylabel("Mean time per round (s)")
plt.title("Hybrid model: F1-score vs training time per configuration")
save("hyperparameter_f1_time.png")

# Confusion matrix for the final Hybrid model (main config, first client count)
best_model = tf.keras.models.load_model(os.path.join(RESULTS_DIR, f"model_Hybrid_{n0}clients_{MAIN_CONFIG}.keras"))
_, prob = evaluate(best_model)
cm = confusion_matrix(y_test, (prob >= THRESHOLD).astype(int))
plt.figure(figsize=(4, 3.5)); plt.imshow(cm, cmap="Blues")
for (i, j), v in np.ndenumerate(cm): plt.text(j, i, f"{v:,}", ha="center", va="center")
plt.xticks([0, 1], ["Normal", "Attack"]); plt.yticks([0, 1], ["Normal", "Attack"])
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Hybrid CNN-LSTM confusion matrix")
save("confusion_matrix_hybrid.png")
print("Figures saved in:", FIG)

## 8. Numbers for the paper
One table with every value the paper reports: final-round metrics, best validation accuracy, the round where 90% validation accuracy was first reached, mean time per round, and model size.

In [ ]:
def rounds_to(df, target=0.90):
    hit = df[df["val_acc"] >= target]
    return int(hit["round"].iloc[0]) if len(hit) else "not reached"

summary = []
for (m, n, c), df in RESULTS.items():
    last = df.iloc[-1]
    params = build_model(m).count_params()
    summary.append({
        "model": m, "clients": n, "config": c,
        "batch_size": CONFIGS[c]["batch_size"], "local_epochs": CONFIGS[c]["local_epochs"], "rounds": CONFIGS[c]["rounds"],
        "final_train_acc": round(last["train_acc"], 4), "final_val_acc": round(last["val_acc"], 4),
        "best_val_acc": round(df["val_acc"].max(), 4),
        "precision": round(last["precision"], 4), "recall": round(last["recall"], 4),
        "f1": round(last["f1"], 4), "roc_auc": round(last["roc_auc"], 4),
        "final_train_loss": round(last["train_loss"], 4), "final_val_loss": round(last["val_loss"], 4),
        "rounds_to_90pct": rounds_to(df), "mean_time_per_round_s": round(df["round_time_s"].mean(), 1),
        "params": params, "update_size_MB": round(params * 4 / 1e6, 2),
    })
summary = pd.DataFrame(summary)
summary.to_csv(os.path.join(RESULTS_DIR, "paper_numbers.csv"), index=False)
pd.set_option("display.max_columns", None)
display(summary)
print("\nSaved:", os.path.join(RESULTS_DIR, "paper_numbers.csv"))
print("Download the whole folder from Google Drive and send me paper_numbers.csv plus the figures folder.")